In [1]:
from itertools import islice
import time

import numpy as np
import pandas as pd

## 1. Wczytanie danych

In [2]:
clicks = pd.concat(
    chunk.loc[chunk["event"] == "click", ["user", "item", "timestamp"]]
    for chunk in pd.read_csv("data/interactions.csv", chunksize=2_000_000)
)

clicks["timestamp"] = pd.to_datetime(clicks["timestamp"], unit="s")

print("liczba kliknięć: ", len(clicks))
print("liczba użytkowników: ", clicks["user"].nunique())
print("liczba ofert: ", clicks["item"].nunique())
print("okres od: ", clicks["timestamp"].min(), " do: ", clicks["timestamp"].max())
clicks.head()

liczba kliknięć:     58817263
liczba użytkowników: 3286174
liczba ofert:        185164
okres:        2020-02-12 00:00:00 - 2020-02-25 23:59:59


,user,item,timestamp
0,27901,56865,2020-02-12
1,124480,115662,2020-02-12
2,159509,5150,2020-02-12
3,188861,109981,2020-02-12
4,207348,88746,2020-02-12


In [2]:
#clicks.to_parquet("data/clicks.parquet", index=False)
clicks = pd.read_parquet("data/clicks.parquet")

## 2. Losowe 10% użytkowników

In [3]:
users = clicks["user"].drop_duplicates().sample(frac=0.1, random_state=42)
sample = clicks[clicks["user"].isin(users)]

print(f"{len(users)} użytkowników, {len(sample)} kliknięć")

328617 użytkowników, 5883756 kliknięć


## 3. Temporal split

In [4]:
t_split = sample["timestamp"].quantile(0.8)

train = sample[sample["timestamp"] < t_split]
test = sample[sample["timestamp"] >= t_split]

print("t_split:", t_split)
print(f"train: {len(train)}, test: {len(test)}")

t_split: 2020-02-22 19:38:10
train: 4707002, test: 1176754


## 4. Katalog: 10% najpopularniejszych ofert

In [5]:
pop = train["item"].value_counts()
catalog = pop.head(int(len(pop) * 0.1)).index

train = train[train["item"].isin(catalog)]
test = test[test["item"].isin(catalog)]

print(f"liczba ofert w katalogu: {len(catalog)}")
print(f"zostało w train: {pop[catalog].sum() / pop.sum():.1%} kliknięć")

print(f"train: {train['user'].nunique()} userów, {train['item'].nunique()} ofert, {len(train)} interakcji")
print(f"test: {test['user'].nunique()} userów, {test['item'].nunique()} ofert, {len(test)} interakcji")

liczba ofert w katalogu: 13317
zostało w train: 44.1% kliknięć
train: 210631 userów, 13317 ofert, 2076316 interakcji
test: 72937 userów, 13257 ofert, 377262 interakcji


In [6]:
# Zostawiamy Ttylko użytkowników, którzy po filtrze katalogu zostali w zbiorze testowym i treningowym. 

train_items = train.groupby("user")["item"].agg(set)
test_items = test.groupby("user")["item"].agg(set)

eval_users = train_items.index.intersection(test_items.index)
test_items = test_items[eval_users]

print(f"liczba użytkowników: {len(eval_users)}")
print(f"średnio ofert w teście: {test_items.map(len).mean():.2f}")

liczba użytkowników: 44316
średnio ofert w teście: 5.07


In [7]:
overlap = [len(test_items[u] & train_items[u]) / len(test_items[u]) for u in eval_users]
print(f"oferty z testu widziane już w train: {np.mean(overlap):.1%}")

oferty z testu widziane już w train: 25.0%


## Metryki

In [8]:
def precision_recall(recs, k=10):
    precision, recall = [], []
    for user in eval_users:
        hits = len(set(recs[user][:k]) & test_items[user])
        precision.append(hits / k)
        recall.append(hits / len(test_items[user]))
    return np.mean(precision), np.mean(recall)

## Baseline

In [9]:
t0 = time.time()
pop_order = train["item"].value_counts().index
fit_time = time.time() - t0


def popularity(users, k=10, exclude_seen=False):
    if not exclude_seen:
        return {u: list(pop_order[:k]) for u in users}
    return {
        u: list(islice((i for i in pop_order if i not in train_items[u]), k))
        for u in users
    }

In [10]:
results = []
for label, exclude_seen in [("nie wykluczamy", False), ("wykluczamy", True)]:
    t0 = time.time()
    recs = popularity(eval_users, exclude_seen=exclude_seen)
    predict_time = time.time() - t0

    precision, recall = precision_recall(recs)
    results.append({
        "oferty z train": label,
        "Precision@10": precision,
        "Recall@10": recall,
        "czas treningu [s]": fit_time,
        "czas predykcji [s]": predict_time,
    })

results = pd.DataFrame(results).round(4)
print(results.to_string(index=False))

oferty z train  Precision@10  Recall@10  czas treningu [s]  czas predykcji [s]
nie wykluczamy        0.0019     0.0032             0.0195              0.0781
    wykluczamy        0.0014     0.0025             0.0195              0.3275


## 5. Prosty eksperyment Implicit ALS

Sprawdzenie działania jednej konfiguracji: 64 czynniki, 20 iteracji, sygnał `log1p(liczba kliknięć)` i ufność `1 + alpha × sygnał`.
Dopuszczamy oferty kliknięte wcześniej. Wynik testowy służy tu do wstępnego sprawdzenia działania; nie dobieramy na nim parametrów.

In [ ]:
from scipy.sparse import coo_matrix
from implicit.cpu.als import AlternatingLeastSquares

als_alpha = 40.0
als_users = pd.Index(train['user'].drop_duplicates())
als_items = pd.Index(train['item'].drop_duplicates())
als_counts = train.groupby(['user', 'item'], sort=False).size().reset_index(name='count')
als_confidence = 1.0 + als_alpha * np.log1p(als_counts['count'].to_numpy(dtype=np.float32))
als_matrix = coo_matrix(
    (als_confidence, (als_users.get_indexer(als_counts['user']),
                      als_items.get_indexer(als_counts['item']))),
    shape=(len(als_users), len(als_items)), dtype=np.float32,
).tocsr()

als_model = AlternatingLeastSquares(
    factors=64, regularization=0.01, iterations=20,
    alpha=1.0, random_state=42, num_threads=4,
)  # Ufność została już policzona w macierzy.
als_t0 = time.perf_counter()
als_model.fit(als_matrix, show_progress=True)
als_fit_time = time.perf_counter() - als_t0
#print(f'Trening ALS: {als_fit_time:.2f} s')

In [ ]:

als_eval_rows = als_users.get_indexer(eval_users)
als_recs = {}
als_t0 = time.perf_counter()
for start in range(0, len(eval_users), 256):
    batch_users = eval_users[start:start + 256]
    batch_rows = als_eval_rows[start:start + 256]
    batch_items, _ = als_model.recommend(
        batch_rows, als_matrix[batch_rows], N=10,
        filter_already_liked_items=False,
    )
    als_recs.update({u: als_items[items].tolist() for u, items in zip(batch_users, batch_items)})
als_predict_time = time.perf_counter() - als_t0
als_precision, als_recall = precision_recall(als_recs)
als_results = pd.DataFrame([{
    'model': 'Implicit ALS', 'Precision@10': als_precision, 'Recall@10': als_recall,
    'czas treningu [s]': als_fit_time, 'czas predykcji [s]': als_predict_time,
}])
print(als_results.round(4).to_string(index=False))

       model  Precision@10  Recall@10  czas treningu [s]  czas predykcji [s]
Implicit ALS        0.0612     0.1493            13.4557              0.6956


## 6. Walidacja hiperparametrów ALS

Dotychczasowy model (`log1p`, alpha=40, factors=64, regularization=0.01, 20 iteracji) jest baseline’em ALS. Poniżej oceniamy tę samą konfigurację na walidacji i porównujemy ją z kandydatami.

Walidacja obejmuje ostatnie 20% okresu treningowego. Podział wykonujemy na `sample` przed filtrowaniem ofert, a katalog 10% najpopularniejszych ofert wybieramy wyłącznie z wcześniejszej części. Nie zawężamy ponownie już przefiltrowanego katalogu `train`. Test nie uczestniczy w wyborze parametrów. Dopuszczamy rekomendowanie ofert wcześniej klikniętych.

Badamy sygnał binarny i log1p, alpha, liczbę czynników, regularyzację i liczbę iteracji. W obrębie konfiguracji kontynuujemy trening tego samego modelu między checkpointami. Plateau Recall@10 służy do wczesnego zatrzymania — nie oznacza matematycznej zbieżności funkcji celu.

In [13]:
from itertools import product
from pathlib import Path
from datetime import datetime, timezone
import json

ALS_BASELINE_PARAMS = dict(signal='log1p', alpha=40.0, factors=64, regularization=0.01)
ALS_SIGNALS = ['binary', 'log1p']
ALS_ALPHAS = [1.0, 10.0, 40.0]
ALS_FACTORS = [32, 64]
ALS_REGULARIZATIONS = [0.01, 0.1]
ALS_CHECKPOINTS = [5, 10, 15, 20, 30, 40, 60]
ALS_PATIENCE = 3
ALS_MIN_ITERATIONS = 20
ALS_MIN_DELTA = 1e-5
ALS_SEED = 42
ALS_BATCH_SIZE = 256

# Dane przed filtrowaniem katalogu; wyłącznie okres wcześniejszy niż test.
als_validation_pool = sample[sample['timestamp'] < t_split]
als_validation_cutoff = als_validation_pool['timestamp'].quantile(0.8)
als_inner_train = als_validation_pool[als_validation_pool['timestamp'] < als_validation_cutoff]
als_validation = als_validation_pool[als_validation_pool['timestamp'] >= als_validation_cutoff]
als_validation_pop = als_inner_train['item'].value_counts()
als_validation_catalog = als_validation_pop.head(max(1, int(len(als_validation_pop) * 0.1))).index
als_inner_train = als_inner_train[als_inner_train['item'].isin(als_validation_catalog)]
als_validation = als_validation[als_validation['item'].isin(als_validation_catalog)]

als_search_dir = Path('results') / datetime.now(timezone.utc).strftime('als_validation_%Y%m%dT%H%M%S%fZ')
als_search_dir.mkdir(parents=True, exist_ok=False)
print('Próg walidacji:', als_validation_cutoff)
print('Interakcje treningowe:', len(als_inner_train), 'walidacyjne:', len(als_validation))
print('Wyniki:', als_search_dir)

Próg walidacji: 2020-02-20 22:37:26
Interakcje treningowe: 1699285 walidacyjne: 316859
Wyniki: results/als_validation_20261015T202536740798Z


In [14]:
def prepare_als_data(interactions, future):
    users = pd.Index(interactions['user'].drop_duplicates())
    items = pd.Index(interactions['item'].drop_duplicates())
    if len(items) < 10:
        raise ValueError('Katalog musi zawierać przynajmniej 10 ofert.')
    counts = interactions.groupby(['user', 'item'], sort=False).size().reset_index(name='count')
    matrix = coo_matrix(
        (counts['count'].to_numpy(dtype=np.float32),
         (users.get_indexer(counts['user']), items.get_indexer(counts['item']))),
        shape=(len(users), len(items)), dtype=np.float32,
    ).tocsr()
    future = future[future['item'].isin(items)]
    targets = future.groupby('user')['item'].agg(set)
    evaluation_users = users.intersection(targets.index, sort=False)
    if evaluation_users.empty:
        raise ValueError('Brak wspólnych użytkowników w treningu i zbiorze oceny.')
    return dict(
        counts=matrix, users=users, items=items,
        eval_rows=users.get_indexer(evaluation_users),
        targets=[set(items.get_indexer(list(targets[u]))) for u in evaluation_users],
    )


def make_als_confidence(counts, signal, alpha):
    if alpha <= 0:
        raise ValueError('alpha musi być dodatnie.')
    matrix = counts.copy()
    if signal == 'binary':
        matrix.data[:] = 1.0
    elif signal == 'log1p':
        np.log1p(matrix.data, out=matrix.data)
    else:
        raise ValueError(f'Nieznany sygnał: {signal}')
    matrix.data *= alpha
    matrix.data += 1.0  # jawna ufność; w modelu alpha=1
    return matrix


def evaluate_als(model, matrix, data):
    precision_sum = recall_sum = recommend_seconds = 0.0
    for start in range(0, len(data['eval_rows']), ALS_BATCH_SIZE):
        rows = data['eval_rows'][start:start + ALS_BATCH_SIZE]
        t0 = time.perf_counter()
        recommendations, _ = model.recommend(
            rows, matrix[rows], N=10, filter_already_liked_items=False,
        )
        recommend_seconds += time.perf_counter() - t0
        for offset, rec in enumerate(recommendations):
            target = data['targets'][start + offset]
            hits = len(set(rec) & target)
            precision_sum += hits / 10
            recall_sum += hits / len(target)
    n = len(data['eval_rows'])
    return {'Precision@10': precision_sum / n, 'Recall@10': recall_sum / n,
            'recommend_seconds': recommend_seconds}


def fit_als(data, params, checkpoints, early_stopping=False):
    checkpoints = list(checkpoints)
    if not checkpoints or checkpoints[0] <= 0 or checkpoints != sorted(set(checkpoints)):
        raise ValueError('Checkpointy muszą być dodatnie i ściśle rosnące.')
    matrix = make_als_confidence(data['counts'], params['signal'], params['alpha'])
    model = AlternatingLeastSquares(
        factors=params['factors'], regularization=params['regularization'],
        alpha=1.0, random_state=ALS_SEED, num_threads=4,
    )
    history, completed, stale = [], 0, 0
    fit_seconds, patience_score = 0.0, -np.inf
    for checkpoint in checkpoints:
        model.iterations = checkpoint - completed
        t0 = time.perf_counter()
        # implicit.cpu.als zachowuje istniejące faktory przy kolejnym fit.
        model.fit(matrix, show_progress=False)
        fit_seconds += time.perf_counter() - t0
        completed = checkpoint
        metrics = evaluate_als(model, matrix, data)
        history.append({**params, 'iterations': completed, **metrics, 'fit_seconds': fit_seconds})
        if metrics['Recall@10'] > patience_score + ALS_MIN_DELTA:
            patience_score, stale = metrics['Recall@10'], 0
        elif completed >= ALS_MIN_ITERATIONS:
            stale += 1
        if early_stopping and completed >= ALS_MIN_ITERATIONS and stale >= ALS_PATIENCE:
            break
    return model, matrix, history


als_validation_data = prepare_als_data(als_inner_train, als_validation)
print('Macierz walidacyjnego treningu:', als_validation_data['counts'].shape)
print('Użytkownicy do oceny:', len(als_validation_data['eval_rows']))

Macierz walidacyjnego treningu: (183481, 13019)
Użytkownicy do oceny: 41636


### Baseline ALS na walidacji i przeszukiwanie konfiguracji

Najpierw trenujemy konfigurację baseline’u od nowa, wyłącznie na wewnętrznym treningu. Następnie sprawdzamy 24 konfiguracje i rosnące checkpointy iteracji. Wybieramy najwyższy Recall@10, następnie Precision@10, a przy remisie mniej iteracji. Limit 60 iteracji bez plateau oznacza, że nie zaobserwowano zatrzymania w tym zakresie.

In [15]:
als_baseline_val_model, als_baseline_val_matrix, als_baseline_val_history = fit_als(
    als_validation_data, ALS_BASELINE_PARAMS, checkpoints=[20],
)
als_baseline_val_result = als_baseline_val_history[-1]
pd.DataFrame([als_baseline_val_result]).to_csv(als_search_dir / 'baseline_validation.csv', index=False)
display(pd.DataFrame([{'model': 'ALS baseline', **als_baseline_val_result}]))
del als_baseline_val_model, als_baseline_val_matrix

,model,signal,alpha,factors,regularization,iterations,Precision@10,Recall@10,recommend_seconds,fit_seconds
0,ALS baseline,log1p,40.0,64,0.01,20,0.057796,0.148268,0.523033,11.517195


In [16]:
als_search_history, als_search_summary = [], []
for signal, alpha, factors, regularization in product(
    ALS_SIGNALS, ALS_ALPHAS, ALS_FACTORS, ALS_REGULARIZATIONS
):
    params = dict(signal=signal, alpha=alpha, factors=factors, regularization=regularization)
    candidate_model, candidate_matrix, history = fit_als(
        als_validation_data, params, ALS_CHECKPOINTS, early_stopping=True,
    )
    als_search_history.extend(history)
    best_checkpoint = min(history, key=lambda r: (-r['Recall@10'], -r['Precision@10'], r['iterations']))
    summary = {**best_checkpoint, 'iterations_run': history[-1]['iterations'],
               'total_fit_seconds': history[-1]['fit_seconds'],
               'stop_reason': 'validation_plateau' if len(history) < len(ALS_CHECKPOINTS) else 'iteration_limit'}
    als_search_summary.append(summary)
    pd.DataFrame(als_search_history).to_csv(als_search_dir / 'validation_history.csv', index=False)
    pd.DataFrame(als_search_summary).to_csv(als_search_dir / 'validation_summary.csv', index=False)
    print(params, 'Recall@10:', round(summary['Recall@10'], 5),
          'najlepsza iteracja:', summary['iterations'])
    del candidate_model, candidate_matrix

# Baseline zawsze pozostaje kandydatem, nawet jeśli early stopping pominąłby jego checkpoint.
als_validation_ranking = pd.DataFrame([als_baseline_val_result, *als_search_summary]).sort_values(
    ['Recall@10', 'Precision@10', 'iterations'], ascending=[False, False, True], kind='stable'
)
display(als_validation_ranking)
als_best_validation = als_validation_ranking.iloc[0]
als_selected_params = dict(
    signal=str(als_best_validation['signal']), alpha=float(als_best_validation['alpha']),
    factors=int(als_best_validation['factors']),
    regularization=float(als_best_validation['regularization']),
)
als_selected_iterations = int(als_best_validation['iterations'])
als_validation_comparison = pd.DataFrame([
    {'model': 'ALS baseline', **als_baseline_val_result},
    {'model': 'ALS selected', **als_best_validation.to_dict()},
])
display(als_validation_comparison)
als_validation_comparison.to_csv(als_search_dir / 'validation_comparison.csv', index=False)
(als_search_dir / 'selected_config.json').write_text(json.dumps({
    **als_selected_params, 'iterations': als_selected_iterations, 'seed': ALS_SEED,
    'validation_cutoff': str(als_validation_cutoff), 'test_cutoff': str(t_split),
    'grid': dict(signals=ALS_SIGNALS, alphas=ALS_ALPHAS, factors=ALS_FACTORS,
                 regularizations=ALS_REGULARIZATIONS, checkpoints=ALS_CHECKPOINTS),
    'early_stopping': dict(patience=ALS_PATIENCE, min_delta=ALS_MIN_DELTA,
                           min_iterations=ALS_MIN_ITERATIONS),
}, indent=2), encoding='utf-8')

{'signal': 'binary', 'alpha': 1.0, 'factors': 32, 'regularization': 0.01} Recall@10: 0.04962 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 1.0, 'factors': 32, 'regularization': 0.1} Recall@10: 0.04959 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 1.0, 'factors': 64, 'regularization': 0.01} Recall@10: 0.08206 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 1.0, 'factors': 64, 'regularization': 0.1} Recall@10: 0.0819 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 10.0, 'factors': 32, 'regularization': 0.01} Recall@10: 0.08316 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 10.0, 'factors': 32, 'regularization': 0.1} Recall@10: 0.08315 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 10.0, 'factors': 64, 'regularization': 0.01} Recall@10: 0.12564 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 10.0, 'factors': 64, 'regularization': 0.1} Recall@10: 0.1257 najlepsza iteracja: 5
{'signal': 'binary', 'alpha': 40.0, 'factors': 32, 'regularization': 0.01} Recall@

,signal,alpha,factors,regularization,iterations,Precision@10,Recall@10,recommend_seconds,fit_seconds,iterations_run,total_fit_seconds,stop_reason
12,binary,40.0,64,0.10,5,0.064581,0.173249,0.579356,3.233877,40.0,23.821360,validation_plateau
24,log1p,40.0,64,0.10,5,0.065021,0.172785,0.512101,2.890093,40.0,23.102298,validation_plateau
11,binary,40.0,64,0.01,5,0.064180,0.172500,0.573074,2.938319,40.0,23.600322,validation_plateau
23,log1p,40.0,64,0.01,5,0.064809,0.172057,0.533165,2.935298,40.0,23.414342,validation_plateau
0,log1p,40.0,64,0.01,20,0.057796,0.148268,0.523033,11.517195,NaN,NaN,NaN
20,log1p,10.0,64,0.10,5,0.053716,0.126926,0.556519,3.122008,40.0,24.170251,validation_plateau
19,log1p,10.0,64,0.01,5,0.053677,0.126783,0.554714,2.902909,40.0,22.848408,validation_plateau
8,binary,10.0,64,0.10,5,0.052973,0.125698,0.549511,2.922151,40.0,23.010274,validation_plateau
7,binary,10.0,64,0.01,5,0.052892,0.125642,0.545056,2.916935,40.0,23.030658,validation_plateau
22,log1p,40.0,32,0.10,5,0.050125,0.123857,0.482936,1.303755,40.0,10.295658,validation_plateau


,model,signal,alpha,factors,regularization,iterations,Precision@10,Recall@10,recommend_seconds,fit_seconds,iterations_run,total_fit_seconds,stop_reason
0,ALS baseline,log1p,40.0,64,0.01,20,0.057796,0.148268,0.523033,11.517195,NaN,NaN,NaN
1,ALS selected,binary,40.0,64,0.10,5,0.064581,0.173249,0.579356,3.233877,40.0,23.82136,validation_plateau


617

### Końcowe porównanie z baseline’em ALS na teście

Trenujemy wybraną konfigurację od początku na całym istniejącym `train`, przez liczbę iteracji wybraną na walidacji. Wynik porównujemy z wcześniejszym baseline’em ALS na tych samych użytkownikach i celach. Nie używamy tego porównania do dalszego strojenia. Model i mapowania identyfikatorów pozostają w zmiennych `als_tuned_model` i `als_final_data`.

In [17]:
als_final_data = prepare_als_data(train, test)
als_tuned_model, als_tuned_matrix, als_tuned_history = fit_als(
    als_final_data, als_selected_params, checkpoints=[als_selected_iterations],
)
als_tuned_metrics = als_tuned_history[-1]
als_test_comparison = pd.DataFrame([
    {'model': 'ALS baseline', **ALS_BASELINE_PARAMS, 'iterations': 20,
     'Precision@10': als_precision, 'Recall@10': als_recall,
     'fit_seconds': als_fit_time, 'recommend_seconds': als_predict_time},
    {'model': 'ALS tuned', **als_tuned_metrics},
])
display(als_test_comparison)
als_test_comparison.to_csv(als_search_dir / 'test_comparison.csv', index=False)

,model,signal,alpha,factors,regularization,iterations,Precision@10,Recall@10,fit_seconds,recommend_seconds
0,ALS baseline,log1p,40.0,64,0.01,20,0.061228,0.149307,13.455688,0.695579
1,ALS tuned,binary,40.0,64,0.10,5,0.068431,0.174453,3.460529,0.622431
